In [1]:
import pandas as pd
import numpy as np

def section(t):
    print("\n" + "="*90); print(t); print("="*90)

fp = r"C:\dev\projects\heat_mortality_analysis\data\processed\eda_ready\tfe\eda_ready_weekly_tfe.parquet"
df = pd.read_parquet(fp).sort_values("week_start")
df["week_start"] = pd.to_datetime(df["week_start"])

pm_cols = [c for c in ["pm10_mean_week","pm10_max_week","pm10_min_week",
                       "pm25_mean_week","pm25_max_week","pm25_min_week",
                       "coverage_pm10_week","coverage_pm25_week","pm10_days_ok","pm25_days_ok",
                       "n_days__air_quality","station"] if c in df.columns]

section("PM COLUMNS PRESENT")
print(pm_cols)


PM COLUMNS PRESENT
['pm10_mean_week', 'pm10_max_week', 'pm10_min_week', 'pm25_mean_week', 'pm25_max_week', 'pm25_min_week', 'coverage_pm10_week', 'coverage_pm25_week', 'pm10_days_ok', 'pm25_days_ok', 'n_days__air_quality', 'station']


In [4]:
# Reliable filter??
section("PM10 COVERAGE (overall)")
display(df[["pm10_mean_week","pm10_max_week","pm10_min_week"]].isna().mean().to_frame("na_pct"))

df["year"] = df["week_start"].dt.year
section("PM10 COVERAGE BY YEAR")
display(df.groupby("year")["pm10_mean_week"].apply(lambda s: s.notna().mean()).to_frame("pm10_coverage"))

section("PM10 QUALITY (coverage/days_ok)")
display(df[["coverage_pm10_week","pm10_days_ok","n_days__air_quality"]].describe().T)


PM10 COVERAGE (overall)


,na_pct
pm10_mean_week,0.129512
pm10_max_week,0.129512
pm10_min_week,0.129512



PM10 COVERAGE BY YEAR


,pm10_coverage
year,
2015,1.000000
2016,0.903846
2017,0.980769
2018,0.792453
2019,1.000000
2020,0.980769
2021,0.596154
2022,0.576923
2023,1.000000



PM10 QUALITY (coverage/days_ok)


,count,mean,std,min,25%,50%,75%,max
coverage_pm10_week,428.0,0.896501,0.250711,0.0,1.0,1.0,1.0,1.0
pm10_days_ok,428.0,5.927570,2.006287,0.0,6.0,7.0,7.0,7.0
n_days__air_quality,428.0,6.649533,1.196629,1.0,7.0,7.0,7.0,7.0


In [5]:
section("DEFINE pm10_ok_week (recommended)")
df["pm10_ok_week"] = (
    df["pm10_mean_week"].notna()
    & (df["coverage_pm10_week"] >= 0.8)
    & (df["pm10_days_ok"] >= 5)
)

display(df["pm10_ok_week"].value_counts().to_frame("weeks"))


DEFINE pm10_ok_week (recommended)


,weeks
pm10_ok_week,
True,342
False,129


In [6]:
section("PM10 stats: ok vs not ok")
cols = ["pm10_mean_week","pm10_max_week","pm10_min_week","coverage_pm10_week","pm10_days_ok"]
display(df.groupby("pm10_ok_week")[cols].describe().T)

section("Missingness driver (years)")
display(df.groupby(["year","pm10_ok_week"]).size().unstack(fill_value=0))


PM10 stats: ok vs not ok


pm10_ok_week                   False       True 
pm10_mean_week     count   68.000000  342.000000
                   mean    26.566252   29.350020
                   std     24.240476   24.052409
                   min      4.589372    3.265787
                   25%     12.601436   13.868903
                   50%     19.144278   21.984836
                   75%     29.475923   36.961513
                   max    126.611111  182.505952
pm10_max_week      count   68.000000  342.000000
                   mean    79.294118  105.614035
                   std    102.603925  134.549525
                   min     10.000000    8.000000
                   25%     27.750000   38.250000
                   50%     41.950000   59.000000
                   75%     81.450000  106.925000
                   max    539.000000  872.000000
pm10_min_week      count   68.000000  342.000000
                   mean     6.022059    5.870175
                   std      7.136031    7.822394
                   min      0.000000    0.000000
                   25%      2.000000    1.000000
                   50%      3.000000    3.000000
                   75%      8.000000    7.000000
                   max     34.000000   54.000000
coverage_pm10_week count   86.000000  342.000000
                   mean     0.551910    0.983152
                   std      0.395945    0.046395
                   min      0.000000    0.833333
                   25%      0.142857    1.000000
                   50%      0.714286    1.000000
                   75%      1.000000    1.000000
                   max      1.000000    1.000000
pm10_days_ok       count   86.000000  342.000000
                   mean     2.383721    6.818713
                   std      1.867169    0.455530
                   min      0.000000    5.000000
                   25%      1.000000    7.000000
                   50%      2.500000    7.000000
                   75%      4.000000    7.000000
                   max      5.000000    7.000000


Missingness driver (years)


pm10_ok_week,False,True
year,,
2015,1,0
2016,23,29
2017,5,47
2018,16,37
2019,6,46
2020,3,49
2021,35,17
2022,34,18
2023,3,49


In [7]:
df["pm10_mean_week_ok"] = df["pm10_mean_week"].where(df["pm10_ok_week"])

In [8]:
good_years = [2017, 2018, 2019, 2020, 2023, 2024]
df_good = df[df["year"].isin(good_years)].copy()

In [10]:

section("CAP columns present")
cap_cols = [c for c in df.columns if c.startswith("cap_") or "cap" in c.lower()]
print(cap_cols)


CAP columns present
['cap_heat_level_max_week', 'cap_dust_level_max_week', 'cap_heat_yellow_plus_week', 'cap_dust_yellow_plus_week', 'cap_coverage_week']


In [11]:
section("CAP dust vs PM10 (only pm10_ok_week==True)")

cap_dust_col = None
for cand in ["cap_dust_yellow_plus_week", "cap_dust_level_max_week"]:
    if cand in df.columns:
        cap_dust_col = cand
        break

print("Using CAP dust column:", cap_dust_col)

d = df[df["pm10_ok_week"]].copy()
thr = d["pm10_mean_week"].quantile(0.80)
d["pm10_high"] = (d["pm10_mean_week"] > thr).astype(int)

if cap_dust_col is None:
    print("No CAP dust column found.")
else:
    if cap_dust_col.endswith("_level_max_week"):
        d["cap_dust_yellow"] = (d[cap_dust_col] >= 2).astype(int)
    else:
        d["cap_dust_yellow"] = d[cap_dust_col].fillna(0).astype(int)

    ct = pd.crosstab(d["cap_dust_yellow"], d["pm10_high"], rownames=["CAP_dust_yellow"], colnames=["PM10_high(q80)"])
    display(ct)

    section("PM10 stats by CAP dust yellow")
    display(d.groupby("cap_dust_yellow")["pm10_mean_week"].describe().T)
    print("PM10 threshold q80 on ok weeks:", float(thr))


CAP dust vs PM10 (only pm10_ok_week==True)
Using CAP dust column: cap_dust_yellow_plus_week


PM10_high(q80),0,1
CAP_dust_yellow,,
0,273,69



PM10 stats by CAP dust yellow


cap_dust_yellow,0
count,342.000000
mean,29.350020
std,24.052409
min,3.265787
25%,13.868903
50%,21.984836
75%,36.961513
max,182.505952


PM10 threshold q80 on ok weeks: 40.79452380952381


In [12]:
section("CAP dust distribution (all weeks)")
display(df["cap_dust_yellow_plus_week"].value_counts(dropna=False).to_frame("weeks"))

if "cap_dust_level_max_week" in df.columns:
    section("CAP dust level distribution (all weeks)")
    display(df["cap_dust_level_max_week"].value_counts(dropna=False).sort_index().to_frame("weeks"))


CAP dust distribution (all weeks)


,weeks
cap_dust_yellow_plus_week,
0.0,471



CAP dust level distribution (all weeks)


,weeks
cap_dust_level_max_week,
0.0,471


In [13]:
section("CAP columns + how many non-zero")

cap_cols = [c for c in df.columns if c.startswith("cap_")]
print(cap_cols)

nonzero = []
for c in cap_cols:
    s = df[c]
    # cuenta valores distintos de 0 (y no nulos)
    nz = int(((s.fillna(0) != 0)).sum())
    nonzero.append((c, nz, df[c].nunique(dropna=False)))

out = pd.DataFrame(nonzero, columns=["col","nonzero_rows","nunique"])
display(out.sort_values(["nonzero_rows","nunique"], ascending=False))


CAP columns + how many non-zero
['cap_heat_level_max_week', 'cap_dust_level_max_week', 'cap_heat_yellow_plus_week', 'cap_dust_yellow_plus_week', 'cap_coverage_week']


,col,nonzero_rows,nunique
4,cap_coverage_week,342,4
0,cap_heat_level_max_week,0,1
1,cap_dust_level_max_week,0,1
2,cap_heat_yellow_plus_week,0,1
3,cap_dust_yellow_plus_week,0,1


In [14]:
for c in ["cap_heat_yellow_plus_week","cap_heat_level_max_week","cap_coverage_week"]:
    if c in df.columns:
        section(f"Distribution: {c}")
        display(df[c].value_counts(dropna=False).sort_index().to_frame("weeks"))


Distribution: cap_heat_yellow_plus_week


,weeks
cap_heat_yellow_plus_week,
0.0,471



Distribution: cap_heat_level_max_week


,weeks
cap_heat_level_max_week,
0.0,471



Distribution: cap_coverage_week


,weeks
cap_coverage_week,
0.000000,129
0.285714,1
0.857143,11
1.000000,330


In [15]:
import pandas as pd
import numpy as np

def section(t):
    print("\n" + "="*90); print(t); print("="*90)

fp = r"C:\dev\projects\heat_mortality_analysis\data\processed\tenerife\cap\cap_alerts_tfe_2018-06-18_2024-12-31.parquet"
cap = pd.read_parquet(fp)

section("GLANCE")
print("shape:", cap.shape)
display(cap.head(10))
display(cap.dtypes.to_frame("dtype"))

section("DATE COVERAGE")
# intenta detectar columna de fecha
date_cols = [c for c in cap.columns if "date" in c.lower() or "day" in c.lower() or "fecha" in c.lower()]
print("candidate date cols:", date_cols)

# si existe algo tipo 'date' o 'day', conviértelo:
for c in date_cols:
    cap[c] = pd.to_datetime(cap[c], errors="coerce")

# si tienes una columna clave tipo 'week_start' ya, también:
if "week_start" in cap.columns:
    cap["week_start"] = pd.to_datetime(cap["week_start"], errors="coerce")

# muestra rango temporal con la mejor columna disponible
for c in ["date","day","fecha","week_start"]:
    if c in cap.columns:
        section(f"RANGE using {c}")
        print(cap[c].min(), "->", cap[c].max())
        break


GLANCE
shape: (390788, 29)


,identifier,sender,sent,status,msgType,scope,event,urgency,severity,certainty,...,onset_dt,expires_dt,year_onset,has_tenerife,has_gran_canaria,has_la_palma,has_lanzarote,has_fuerteventura,has_la_gomera,has_el_hierro
0,2.49.0.0.724.0.ES.20180617215001.65VV65ATTA202...,http://www.aemet.es,2018-06-17T21:50:01-00:00,Actual,Alert,Public,Aviso de temperaturas máximas de nivel verde,Future,Minor,Likely,...,2018-06-19 23:00:00+00:00,2018-06-20 22:59:59+00:00,2018,1,1,1,1,1,1,1
1,2.49.0.0.724.0.ES.20180617215001.65VV65ATTA202...,http://www.aemet.es,2018-06-17T21:50:01-00:00,Actual,Alert,Public,Minor high-temperature warning,Future,Minor,Likely,...,2018-06-19 23:00:00+00:00,2018-06-20 22:59:59+00:00,2018,1,1,1,1,1,1,1
2,2.49.0.0.724.0.ES.20180617215001.65VV65BTTI202...,http://www.aemet.es,2018-06-17T21:50:01-00:00,Actual,Alert,Public,Aviso de temperaturas mínimas de nivel verde,Future,Minor,Likely,...,2018-06-19 23:00:00+00:00,2018-06-20 22:59:59+00:00,2018,1,1,1,1,1,1,1
3,2.49.0.0.724.0.ES.20180617215001.65VV65BTTI202...,http://www.aemet.es,2018-06-17T21:50:01-00:00,Actual,Alert,Public,Minor low-temperature warning,Future,Minor,Likely,...,2018-06-19 23:00:00+00:00,2018-06-20 22:59:59+00:00,2018,1,1,1,1,1,1,1
4,2.49.0.0.724.0.ES.20180617215001.65VV65COCO202...,http://www.aemet.es,2018-06-17T21:50:01-00:00,Actual,Alert,Public,Aviso de costeros de nivel verde,Future,Minor,Likely,...,2018-06-19 23:00:00+00:00,2018-06-20 22:59:59+00:00,2018,1,1,1,1,1,1,1
5,2.49.0.0.724.0.ES.20180617215001.65VV65COCO202...,http://www.aemet.es,2018-06-17T21:50:01-00:00,Actual,Alert,Public,Minor coastalevent warning,Future,Minor,Likely,...,2018-06-19 23:00:00+00:00,2018-06-20 22:59:59+00:00,2018,1,1,1,1,1,1,1
6,2.49.0.0.724.0.ES.20180617215001.65VV65NENV202...,http://www.aemet.es,2018-06-17T21:50:01-00:00,Actual,Alert,Public,Aviso de nevadas de nivel verde,Future,Minor,Likely,...,2018-06-19 23:00:00+00:00,2018-06-20 22:59:59+00:00,2018,1,1,1,1,1,1,1
7,2.49.0.0.724.0.ES.20180617215001.65VV65NENV202...,http://www.aemet.es,2018-06-17T21:50:01-00:00,Actual,Alert,Public,Minor snow warning,Future,Minor,Likely,...,2018-06-19 23:00:00+00:00,2018-06-20 22:59:59+00:00,2018,1,1,1,1,1,1,1
8,2.49.0.0.724.0.ES.20180617215001.65VV65NINI202...,http://www.aemet.es,2018-06-17T21:50:01-00:00,Actual,Alert,Public,Aviso de nieblas de nivel verde,Future,Minor,Likely,...,2018-06-19 23:00:00+00:00,2018-06-20 22:59:59+00:00,2018,1,1,1,1,1,1,1
9,2.49.0.0.724.0.ES.20180617215001.65VV65NINI202...,http://www.aemet.es,2018-06-17T21:50:01-00:00,Actual,Alert,Public,Minor fog warning,Future,Minor,Likely,...,2018-06-19 23:00:00+00:00,2018-06-20 22:59:59+00:00,2018,1,1,1,1,1,1,1


,dtype
identifier,object
sender,object
sent,object
status,object
msgType,object
scope,object
event,object
urgency,object
severity,object
certainty,object



DATE COVERAGE
candidate date cols: []


In [16]:
section("CANDIDATE CAP TYPE/LEVEL COLUMNS")
cap_cols = [c for c in cap.columns if any(k in c.lower() for k in ["dust","calima","heat","temp","level","yellow","orange","red","phenomenon","fenomeno","tipo","risk","riesgo","aviso","alert"])]
print(cap_cols)

section("NON-ZERO / UNIQUE COUNTS (quick audit)")
rows = []
for c in cap_cols:
    s = cap[c]
    # intenta numérico si parece nivel
    if s.dtype == "O":
        # no conviertas todo, solo para conteo simple
        pass
    nonzero = int(((s.fillna(0) != 0)).sum()) if pd.api.types.is_numeric_dtype(s) else None
    rows.append({"col": c, "dtype": str(s.dtype), "nunique": int(s.nunique(dropna=False)), "nonzero_rows": nonzero})

audit = pd.DataFrame(rows).sort_values(["nonzero_rows","nunique"], ascending=False)
display(audit)


CANDIDATE CAP TYPE/LEVEL COLUMNS
[]

NON-ZERO / UNIQUE COUNTS (quick audit)


KeyError: 'nonzero_rows'

In [17]:
section("ALL COLUMNS")
print(cap.columns.tolist())


ALL COLUMNS
['identifier', 'sender', 'sent', 'status', 'msgType', 'scope', 'event', 'urgency', 'severity', 'certainty', 'onset', 'expires', 'headline', 'description', 'areaDesc', 'cap_file', 'sh_url', 'range_start', 'range_end', 'onset_dt', 'expires_dt', 'year_onset', 'has_tenerife', 'has_gran_canaria', 'has_la_palma', 'has_lanzarote', 'has_fuerteventura', 'has_la_gomera', 'has_el_hierro']


In [18]:
section("GENERIC COLUMN AUDIT (top 50)")
rows = []
for c in cap.columns:
    s = cap[c]
    nunq = int(s.nunique(dropna=False))
    nonnull = int(s.notna().sum())
    rows.append({"col": c, "dtype": str(s.dtype), "nonnull": nonnull, "nunique": nunq})

audit = pd.DataFrame(rows).sort_values(["nunique","nonnull"], ascending=False)
display(audit.head(50))


GENERIC COLUMN AUDIT (top 50)


,col,dtype,nonnull,nunique
15,cap_file,object,390788,195394
0,identifier,object,390788,24675
2,sent,object,390788,3469
10,onset,object,390788,3141
19,onset_dt,"datetime64[ns, UTC]",390788,3141
11,expires,object,390788,2976
20,expires_dt,"datetime64[ns, UTC]",390788,2976
13,description,object,66622,2205
12,headline,object,390788,84
6,event,object,390788,42


In [19]:
section("SEARCH KEYWORDS IN COLUMN NAMES")
keywords = ["dust","calima","sahara","aemet","cap","heat","temp","aviso","alert","level","yellow","risk","fenomen"]
for k in keywords:
    hits = [c for c in cap.columns if k in c.lower()]
    if hits:
        print(k, "->", hits)


SEARCH KEYWORDS IN COLUMN NAMES
cap -> ['cap_file']


In [21]:
cap.shape


(390788, 29)

In [22]:
section("CAP events (overall)")
display(cap["event"].value_counts().head(30).to_frame("rows"))

section("Tenerife flag distribution")
display(cap["has_tenerife"].value_counts(dropna=False).to_frame("rows"))


CAP events (overall)


,rows
event,
Aviso de temperaturas máximas de nivel verde,17923
Minor high-temperature warning,17923
Aviso de temperaturas mínimas de nivel verde,17923
Minor low-temperature warning,17923
Aviso de costeros de nivel verde,17923
Minor coastalevent warning,17923
Aviso de nevadas de nivel verde,17923
Minor snow warning,17923
Aviso de nieblas de nivel verde,17923



Tenerife flag distribution


,rows
has_tenerife,
1,390788


In [23]:
import pandas as pd

cap_fp = r"C:\dev\projects\heat_mortality_analysis\data\processed\tenerife\cap\cap_weekly_tfe_2018_2024.parquet"
cap_year = pd.read_parquet(cap_fp)

cap_year.columns.to_list()

['week_start',
 'cap_heat_level_max_week',
 'cap_dust_level_max_week',
 'cap_heat_yellow_plus_week',
 'cap_dust_yellow_plus_week',
 'cap_coverage_week']

In [24]:
section("CAP weekly audit vs df (EDA-ready)")

capw = pd.read_parquet(r"C:\dev\projects\heat_mortality_analysis\data\processed\tenerife\cap\cap_weekly_tfe_2018_2024.parquet")
capw["week_start"] = pd.to_datetime(capw["week_start"])
df["week_start"] = pd.to_datetime(df["week_start"])

print("capw range:", capw["week_start"].min(), "->", capw["week_start"].max(), "| rows:", len(capw))
print("df   range:", df["week_start"].min(), "->", df["week_start"].max(), "| rows:", len(df))

# ¿se alinean semanas?
a = set(df["week_start"])
b = set(capw["week_start"])
print("weeks in capw not in df:", len(b - a))
print("weeks in df not in capw:", len(a - b))

# merge rápido y ver si aparecen valores !=0
tmp = df[["week_start"]].merge(capw, on="week_start", how="left")
print("nonzero dust weeks in capw:", int((capw["cap_dust_level_max_week"]>0).sum()))
print("nonzero dust weeks after merge:", int((tmp["cap_dust_level_max_week"]>0).sum()))

display(capw["cap_dust_level_max_week"].value_counts().sort_index().to_frame("capw"))
display(tmp["cap_dust_level_max_week"].value_counts(dropna=False).sort_index().to_frame("after_merge"))


CAP weekly audit vs df (EDA-ready)
capw range: 2018-06-18 00:00:00 -> 2024-12-30 00:00:00 | rows: 342
df   range: 2015-12-28 00:00:00 -> 2024-12-30 00:00:00 | rows: 471
weeks in capw not in df: 0
weeks in df not in capw: 129
nonzero dust weeks in capw: 0
nonzero dust weeks after merge: 0


,capw
cap_dust_level_max_week,
0,342


,after_merge
cap_dust_level_max_week,
0.0,342
NaN,129


In [25]:
import pandas as pd

p = r"C:\dev\projects\heat_mortality_analysis\data\processed\tenerife\cap\cap_weekly_tfe_start_end.parquet"
capw = pd.read_parquet(p)

print("rows:", len(capw), "range:", capw["week_start"].min(), "->", capw["week_start"].max())
print("\nDust level counts:")
print(capw["cap_dust_level_max_week"].value_counts().sort_index().head(10))
print("\nDust yellow+ counts:")
print(capw["cap_dust_yellow_plus_week"].value_counts(dropna=False))
print("\nHeat level counts:")
print(capw["cap_heat_level_max_week"].value_counts().sort_index().head(10))

rows: 342 range: 2018-06-18 00:00:00+00:00 -> 2024-12-30 00:00:00+00:00

Dust level counts:
cap_dust_level_max_week
0    342
Name: count, dtype: int64

Dust yellow+ counts:
cap_dust_yellow_plus_week
0    342
Name: count, dtype: int64

Heat level counts:
cap_heat_level_max_week
0    342
Name: count, dtype: int64


In [26]:
import pandas as pd
from pathlib import Path

base = Path(r"C:\dev\projects\heat_mortality_analysis\data\interim\cap\canarias\year_onset=2021")
fps = sorted(base.rglob("*.parquet"))

# carga solo 1–3 parquets para ir rápido
raw = pd.concat(
    [pd.read_parquet(fps[i], columns=["event","headline","severity","areaDesc","onset_dt"]) 
     for i in range(min(3, len(fps)))],
    ignore_index=True
)

tfe = raw[raw["areaDesc"].str.contains("Tenerife", case=False, na=False)].copy()
print("rows Tenerife:", len(tfe))

print("\nTop events:")
print(tfe["event"].value_counts().head(20))

print("\nSeverity values:")
print(tfe["severity"].value_counts(dropna=False).head(20))

print("\nHeadline sample:")
print(tfe["headline"].dropna().head(10).to_list())

rows Tenerife: 12572

Top events:
event
Moderate rain warning                           659
Aviso de lluvias de nivel amarillo              659
Aviso de costeros de nivel amarillo             577
Moderate coastalevent warning                   577
Aviso de vientos de nivel amarillo              542
Moderate wind warning                           542
Aviso de nevadas de nivel verde                 445
Minor snow warning                              445
Aviso de temperaturas máximas de nivel verde    445
Minor high-temperature warning                  445
Aviso de temperaturas mínimas de nivel verde    445
Minor low-temperature warning                   445
Aviso de costeros de nivel verde                445
Minor coastalevent warning                      445
Minor thunderstorm warning                      445
Aviso de tormentas de nivel verde               445
Minor rain warning                              445
Aviso de lluvias de nivel verde                 445
Minor fog warning       

In [27]:
DUST_RE = r"(dust|polvo)"
HEAT_RE = r"(temper|high-temperature|calor)"

tfe["is_dust_event"] = (
    tfe["event"].fillna("").str.contains(DUST_RE, case=False)
    | tfe["headline"].fillna("").str.contains(DUST_RE, case=False)
).astype(int)

tfe["is_heat_event"] = (
    tfe["event"].fillna("").str.contains(HEAT_RE, case=False)
    | tfe["headline"].fillna("").str.contains(HEAT_RE, case=False)
).astype(int)

SEV_MAP = {"minor": 1, "moderate": 2, "severe": 3, "extreme": 4}
tfe["sev_norm"] = tfe["severity"].fillna("").astype(str).str.strip().str.lower()
tfe["level_score"] = tfe["sev_norm"].map(SEV_MAP).fillna(0).astype(int)

print("dust rows:", int(tfe["is_dust_event"].sum()))
print("heat rows:", int(tfe["is_heat_event"].sum()))
print("severity nonzero:", int((tfe["level_score"] > 0).sum()))

print("\nUnique severity -> level_score:")
print(tfe[["severity","sev_norm","level_score"]].drop_duplicates().head(30))

dust rows: 890
heat rows: 1780
severity nonzero: 12572

Unique severity -> level_score:
     severity  sev_norm  level_score
0       Minor     minor            1
818    Severe    severe            3
844  Moderate  moderate            2


C:\Users\fdora\AppData\Local\Temp\ipykernel_22028\437827668.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  tfe["event"].fillna("").str.contains(DUST_RE, case=False)
C:\Users\fdora\AppData\Local\Temp\ipykernel_22028\437827668.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  | tfe["headline"].fillna("").str.contains(DUST_RE, case=False)
C:\Users\fdora\AppData\Local\Temp\ipykernel_22028\437827668.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  tfe["event"].fillna("").str.contains(HEAT_RE, case=False)
C:\Users\fdora\AppData\Local\Temp\ipykernel_22028\437827668.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  | tfe["headline

In [28]:
print("rows Tenerife:", len(tfe))
print(tfe["event"].value_counts().head(20))
print(tfe["severity"].value_counts(dropna=False).head(20))

print("dust rows:", int(tfe["is_dust_event"].sum()))
print("heat rows:", int(tfe["is_heat_event"].sum()))
print("severity nonzero:", int((tfe["level_score"] > 0).sum()))
print(tfe[["severity","sev_norm","level_score"]].drop_duplicates().head(30))

rows Tenerife: 12572
event
Moderate rain warning                           659
Aviso de lluvias de nivel amarillo              659
Aviso de costeros de nivel amarillo             577
Moderate coastalevent warning                   577
Aviso de vientos de nivel amarillo              542
Moderate wind warning                           542
Aviso de nevadas de nivel verde                 445
Minor snow warning                              445
Aviso de temperaturas máximas de nivel verde    445
Minor high-temperature warning                  445
Aviso de temperaturas mínimas de nivel verde    445
Minor low-temperature warning                   445
Aviso de costeros de nivel verde                445
Minor coastalevent warning                      445
Minor thunderstorm warning                      445
Aviso de tormentas de nivel verde               445
Minor rain warning                              445
Aviso de lluvias de nivel verde                 445
Minor fog warning                    

In [29]:
import pandas as pd
import numpy as np

def section(t):
    print("\n" + "="*90); print(t); print("="*90)

cap_fp = r"C:\dev\projects\heat_mortality_analysis\data\processed\tenerife\cap\cap_weekly_tfe_2018_2024.parquet"
capw = pd.read_parquet(cap_fp).sort_values("week_start")
capw["week_start"] = pd.to_datetime(capw["week_start"], errors="coerce")

section("GLANCE")
print("rows:", len(capw), "range:", capw["week_start"].min(), "->", capw["week_start"].max())
display(capw.head())

section("DISTRIBUTIONS")
for c in ["cap_dust_level_max_week","cap_dust_yellow_plus_week",
          "cap_heat_level_max_week","cap_heat_yellow_plus_week"]:
    if c in capw.columns:
        print(f"\n{c}")
        display(capw[c].value_counts(dropna=False).sort_index().to_frame("weeks"))

section("EVENT WEEKS (examples)")
display(capw[capw["cap_dust_level_max_week"] >= 1][["week_start","cap_dust_level_max_week","cap_dust_yellow_plus_week","cap_coverage_week"]].head(15))
display(capw[capw["cap_dust_level_max_week"] >= 2][["week_start","cap_dust_level_max_week","cap_dust_yellow_plus_week","cap_coverage_week"]].head(15))

section("COVERAGE")
if "cap_coverage_week" in capw.columns:
    display(capw["cap_coverage_week"].describe().to_frame().T)
    print("weeks with coverage==0:", int((capw["cap_coverage_week"] == 0).sum()))


GLANCE
rows: 342 range: 2018-06-18 00:00:00 -> 2024-12-30 00:00:00


,week_start,cap_heat_level_max_week,cap_dust_level_max_week,cap_heat_yellow_plus_week,cap_dust_yellow_plus_week,cap_coverage_week
0,2018-06-18,1,1,0,0,0.857143
1,2018-06-25,1,1,0,0,1.000000
2,2018-07-02,1,1,0,0,1.000000
3,2018-07-09,1,1,0,0,1.000000
4,2018-07-16,1,1,0,0,1.000000



DISTRIBUTIONS

cap_dust_level_max_week


,weeks
cap_dust_level_max_week,
1,323
2,19



cap_dust_yellow_plus_week


...

,week_start,cap_dust_level_max_week,cap_dust_yellow_plus_week,cap_coverage_week
0,2018-06-18,1,0,0.857143
1,2018-06-25,1,0,1.000000
2,2018-07-02,1,0,1.000000
3,2018-07-09,1,0,1.000000
4,2018-07-16,1,0,1.000000
5,2018-07-23,1,0,1.000000
6,2018-07-30,1,0,1.000000
7,2018-08-06,1,0,1.000000
8,2018-08-13,1,0,0.857143
9,2018-08-20,1,0,1.000000


,week_start,cap_dust_level_max_week,cap_dust_yellow_plus_week,cap_coverage_week
26,2018-12-17,2,1,1.0
27,2018-12-24,2,1,1.0
85,2020-02-03,2,1,1.0
87,2020-02-17,2,1,1.0
139,2021-02-15,2,1,1.0
164,2021-08-09,2,1,1.0
185,2022-01-03,2,1,1.0
186,2022-01-10,2,1,1.0
187,2022-01-17,2,1,1.0
188,2022-01-24,2,1,1.0



COVERAGE


,count,mean,std,min,25%,50%,75%,max
cap_coverage_week,342.0,0.993317,0.045932,0.285714,1.0,1.0,1.0,1.0


weeks with coverage==0: 0


In [30]:
import pandas as pd

def section(t):
    print("\n" + "="*90); print(t); print("="*90)

cap_fp = r"C:\dev\projects\heat_mortality_analysis\data\processed\tenerife\cap\cap_weekly_tfe_2018_2024.parquet"
capw = pd.read_parquet(cap_fp).sort_values("week_start")
capw["week_start"] = pd.to_datetime(capw["week_start"])

capw["year"] = capw["week_start"].dt.year

section("COVERAGE BY YEAR (counts + event rates)")
by_year = capw.groupby("year").agg(
    n_weeks=("week_start","size"),
    dust_yellow_weeks=("cap_dust_yellow_plus_week","sum"),
    heat_yellow_weeks=("cap_heat_yellow_plus_week","sum"),
    dust_any_weeks=("cap_dust_level_max_week", lambda s: int((s >= 1).sum())),
    heat_any_weeks=("cap_heat_level_max_week", lambda s: int((s >= 1).sum())),
    coverage_mean=("cap_coverage_week","mean"),
    coverage_min=("cap_coverage_week","min"),
)
by_year["dust_yellow_rate"] = by_year["dust_yellow_weeks"] / by_year["n_weeks"]
by_year["heat_yellow_rate"] = by_year["heat_yellow_weeks"] / by_year["n_weeks"]
display(by_year)

section("YEARS WITH SUSPICIOUSLY LOW COVERAGE")
display(by_year[(by_year["coverage_mean"] < 0.5) | (by_year["coverage_min"] == 0)])


COVERAGE BY YEAR (counts + event rates)


,n_weeks,dust_yellow_weeks,heat_yellow_weeks,dust_any_weeks,heat_any_weeks,coverage_mean,coverage_min,dust_yellow_rate,heat_yellow_rate
year,,,,,,,,,
2018,29,2,1,29,29,0.980296,0.857143,0.068966,0.034483
2019,52,0,3,52,52,0.997253,0.857143,0.000000,0.057692
2020,52,2,7,52,52,0.997253,0.857143,0.038462,0.134615
2021,52,2,8,52,52,0.997253,0.857143,0.038462,0.153846
2022,52,7,7,52,52,0.994505,0.857143,0.134615,0.134615
2023,52,4,9,52,52,0.997253,0.857143,0.076923,0.173077
2024,53,2,7,53,53,0.983827,0.285714,0.037736,0.132075



YEARS WITH SUSPICIOUSLY LOW COVERAGE


,n_weeks,dust_yellow_weeks,heat_yellow_weeks,dust_any_weeks,heat_any_weeks,coverage_mean,coverage_min,dust_yellow_rate,heat_yellow_rate
year,,,,,,,,,


In [31]:
low = capw[(capw["year"]==2024) & (capw["cap_coverage_week"] <= 0.3)]
low[["week_start","cap_coverage_week","cap_heat_level_max_week","cap_dust_level_max_week"]]

,week_start,cap_coverage_week,cap_heat_level_max_week,cap_dust_level_max_week
341,2024-12-30,0.285714,1,1


In [32]:
section("PM10 missing vs quality (2021–2022 focus)")
focus = df[df["year"].isin([2021, 2022])].copy()

display(focus[["pm10_mean_week","coverage_pm10_week","pm10_days_ok","n_days__air_quality","station"]].head(20))
display(focus["station"].value_counts(dropna=False).head(20).to_frame("rows"))

section("Is PM10 missing or just low-quality?")
print("pm10_mean_week NA pct:", float(focus["pm10_mean_week"].isna().mean()))
print("coverage_pm10_week NA pct:", float(focus["coverage_pm10_week"].isna().mean()))
print("pm10_days_ok NA pct:", float(focus["pm10_days_ok"].isna().mean()))


PM10 missing vs quality (2021–2022 focus)


,pm10_mean_week,coverage_pm10_week,pm10_days_ok,n_days__air_quality,station
262,22.009354,1.000000,7.0,7.0,casa_cuna
263,12.208333,1.000000,1.0,1.0,casa_cuna
264,NaN,NaN,NaN,NaN,None
265,7.416667,1.000000,1.0,1.0,casa_cuna
266,21.708333,0.857143,6.0,7.0,casa_cuna
267,24.770833,1.000000,4.0,4.0,casa_cuna
268,NaN,NaN,NaN,NaN,None
269,10.791667,1.000000,1.0,1.0,casa_cuna
270,24.065476,1.000000,7.0,7.0,casa_cuna
271,22.729167,1.000000,4.0,4.0,casa_cuna


,rows
station,
casa_cuna,61
None,43



Is PM10 missing or just low-quality?
pm10_mean_week NA pct: 0.41346153846153844
coverage_pm10_week NA pct: 0.41346153846153844
pm10_days_ok NA pct: 0.41346153846153844


In [33]:
section("If PM10 exists, what coverage/days_ok does it have? (2021–2022)")
tmp = focus[focus["pm10_mean_week"].notna()]
display(tmp[["coverage_pm10_week","pm10_days_ok","n_days__air_quality"]].describe().T)


If PM10 exists, what coverage/days_ok does it have? (2021–2022)


,count,mean,std,min,25%,50%,75%,max
coverage_pm10_week,61.0,0.975800,0.055241,0.833333,1.0,1.0,1.0,1.0
pm10_days_ok,61.0,4.524590,2.078514,1.000000,3.0,5.0,6.0,7.0
n_days__air_quality,61.0,4.688525,2.210136,1.000000,3.0,5.0,7.0,7.0


In [34]:
section("PM10 missing runs (2021–2022)")
focus = df[df["year"].isin([2021, 2022])].sort_values("week_start").copy()
focus["pm10_missing"] = focus["pm10_mean_week"].isna().astype(int)
display(focus[["week_start","pm10_missing","station"]].head(80))
display(focus[["week_start","pm10_missing","station"]].tail(80))

# rachas
focus["grp"] = (focus["pm10_missing"].ne(focus["pm10_missing"].shift())).cumsum()
runs = focus.groupby(["grp","pm10_missing"]).agg(
    start=("week_start","min"),
    end=("week_start","max"),
    weeks=("week_start","size")
).reset_index()
display(runs[runs["pm10_missing"]==1].sort_values("weeks", ascending=False).head(15))


PM10 missing runs (2021–2022)


,week_start,pm10_missing,station
262,2021-01-04,0,casa_cuna
263,2021-01-11,0,casa_cuna
264,2021-01-18,1,None
265,2021-01-25,0,casa_cuna
266,2021-02-01,0,casa_cuna
...,...,...,...
337,2022-06-13,1,None
338,2022-06-20,1,None
339,2022-06-27,0,casa_cuna
340,2022-07-04,0,casa_cuna


,week_start,pm10_missing,station
286,2021-06-21,1,None
287,2021-06-28,0,casa_cuna
288,2021-07-05,0,casa_cuna
289,2021-07-12,1,None
290,2021-07-19,1,None
...,...,...,...
361,2022-11-28,0,casa_cuna
362,2022-12-05,0,casa_cuna
363,2022-12-12,1,None
364,2022-12-19,1,None


,grp,pm10_missing,start,end,weeks
7,8,1,2021-04-12,2021-04-19,2
5,6,1,2021-03-15,2021-03-22,2
11,12,1,2021-06-14,2021-06-21,2
9,10,1,2021-05-17,2021-05-24,2
13,14,1,2021-07-12,2021-07-19,2
15,16,1,2021-08-16,2021-08-23,2
21,22,1,2021-11-15,2021-11-22,2
17,18,1,2021-09-13,2021-09-20,2
45,46,1,2022-11-14,2022-11-21,2
43,44,1,2022-10-17,2022-10-24,2


In [35]:
section("Station vs missing (2021–2022)")
display(pd.crosstab(focus["station"], focus["pm10_missing"], dropna=False))


Station vs missing (2021–2022)


pm10_missing,0,1
station,,
casa_cuna,61,0
NaN,0,43


In [36]:
section("Pattern check: every other week?")
focus = df[df["year"].isin([2021, 2022])].sort_values("week_start").copy()
focus["pm10_missing"] = focus["pm10_mean_week"].isna().astype(int)
focus["week_idx"] = np.arange(len(focus))
display(pd.crosstab(focus["week_idx"] % 2, focus["pm10_missing"]))


Pattern check: every other week?


pm10_missing,0,1
week_idx,,
0,30,22
1,31,21


In [37]:
section("Are other pollutants missing on the same weeks?")
cols = ["pm10_mean_week","pm25_mean_week","no2_mean_week","o3_mean_week","so2_mean_week"]
miss = focus[cols].isna().mean().to_frame("na_pct")
display(miss)

# correlación de missingness
m = focus[cols].isna().astype(int)
display(m.corr())


Are other pollutants missing on the same weeks?


,na_pct
pm10_mean_week,0.413462
pm25_mean_week,0.413462
no2_mean_week,0.413462
o3_mean_week,0.413462
so2_mean_week,0.413462


,pm10_mean_week,pm25_mean_week,no2_mean_week,o3_mean_week,so2_mean_week
pm10_mean_week,1.0,1.0,1.0,1.0,1.0
pm25_mean_week,1.0,1.0,1.0,1.0,1.0
no2_mean_week,1.0,1.0,1.0,1.0,1.0
o3_mean_week,1.0,1.0,1.0,1.0,1.0
so2_mean_week,1.0,1.0,1.0,1.0,1.0


In [2]:
import pandas as pd

fp = r"C:\dev\projects\heat_mortality_analysis\data\processed\tenerife\weather\weather_weekly_tfe_2016_2024.parquet"
df = pd.read_parquet(fp)
print(df.columns.tolist())

['week_start', 'n_days', 'temp_c_mean', 'tmax_c_mean', 'tmax_c_max', 'tmin_c_mean', 'tmin_c_min', 'humidity_mean', 'pressure_hpa_mean', 'wind_ms_mean', 'prec_sum', 'coverage']
